# Experiment 9: Transfer Learning using Pretrained CNN

## Step 1
Import the required libraries, including torchvision models.

In [1]:
import os
import gc
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
import kagglehub

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Step 2
Load the dataset and prepare the images (resize to 64 x 64, 3 channels, normalise).

In [2]:
path = kagglehub.dataset_download("puneet6060/intel-image-classification")


def find_dir(name):
    for root, dirs, files in os.walk(path):
        if os.path.basename(root) == name and "buildings" in dirs:
            return root


train_dir = find_dir("seg_train")
test_dir = find_dir("seg_test")

img_size = 128
batch_size = 32
mean = [0.485, 0.456, 0.406]
std = [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])
eval_tf = transforms.Compose([
    transforms.Resize((img_size, img_size)),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
])

train_full = datasets.ImageFolder(train_dir, train_tf)
val_full = datasets.ImageFolder(train_dir, eval_tf)
test_ds = datasets.ImageFolder(test_dir, eval_tf)
perm = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(42)).tolist()
n_val = int(0.1 * len(perm))
train_ds = Subset(train_full, perm[n_val:])
val_ds = Subset(val_full, perm[:n_val])
class_names = train_full.classes


def make_loader(ds, bs, shuffle):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=device.type == "cuda")


print(class_names)
print(len(train_ds), len(val_ds), len(test_ds))

100%|██████████| 346M/346M [09:30<00:00, 637kB/s] 

Extracting files...


['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']
12631 1403 3000


## Step 3
Load the pretrained ResNet18 and replace its last layer with a layer of 10 outputs.

In [3]:
try:
    model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    print("loaded ImageNet pretrained ResNet18")
except Exception as e:
    print(f"pretrained weights unavailable ({e}); using random initialisation")
    model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)

criterion = nn.CrossEntropyLoss()
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)
opt = None
train_modules = []

loaded ImageNet pretrained ResNet18


## Step 4
Train in two phases: first only the last layer with the other layers frozen, then the last block and the last layer with a small learning rate.

In [4]:
def free_memory():
    if device.type == "cuda":
        torch.cuda.empty_cache()


def is_oom(e):
    return isinstance(e, torch.cuda.OutOfMemoryError) or "out of memory" in str(e).lower()


def safe_call(fn, bs, min_bs=8):
    while True:
        try:
            return fn(bs), bs
        except RuntimeError as e:
            if not is_oom(e):
                raise
        model.zero_grad(set_to_none=True)
        gc.collect()
        free_memory()
        if bs <= min_bs:
            raise RuntimeError("out of memory even at the smallest batch size")
        bs = max(min_bs, bs // 2)
        print(f"out of memory, retrying with batch size {bs}")


def train_one_epoch(bs):
    model.eval()
    for m in train_modules:
        m.train()
    tot = 0.0
    n = 0
    for x, y in make_loader(train_ds, bs, True):
        x, y = x.to(device), y.to(device)
        opt.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            out = model(x)
        loss = criterion(out.float(), y)
        scaler.scale(loss).backward()
        scaler.step(opt)
        scaler.update()
        tot += loss.item() * len(y)
        n += len(y)
    return tot / n


def predict_set(ds, bs):
    model.eval()
    preds = []
    labels = []
    with torch.no_grad():
        for x, y in make_loader(ds, bs, False):
            with torch.autocast(device_type=device.type, enabled=use_amp):
                preds.append(model(x.to(device)).argmax(1).cpu())
            labels.append(y)
    return torch.cat(preds), torch.cat(labels)


def run_phase(name, n_epochs, params, lr, modules):
    global opt, train_modules, batch_size
    opt = torch.optim.Adam(params, lr=lr)
    train_modules = modules
    print(name)
    for ep in range(n_epochs):
        loss, batch_size = safe_call(train_one_epoch, batch_size)
        free_memory()
        (p, l), _ = safe_call(lambda bs: predict_set(val_ds, bs), batch_size)
        print(f"epoch {ep + 1}/{n_epochs} batch_size {batch_size} loss {loss:.4f} val_acc {(p == l).float().mean().item():.4f}")


for p in model.parameters():
    p.requires_grad = False
for p in model.fc.parameters():
    p.requires_grad = True
run_phase("phase 1: backbone frozen, training last layer", 3, model.fc.parameters(), 1e-3, [model.fc])

for p in model.layer4.parameters():
    p.requires_grad = True
params = list(model.layer4.parameters()) + list(model.fc.parameters())
run_phase("phase 2: fine-tuning layer4 and last layer", 3, params, 1e-4, [model.layer4, model.fc])

phase 1: backbone frozen, training last layer
epoch 1/3 batch_size 32 loss 0.4058 val_acc 0.8952
epoch 2/3 batch_size 32 loss 0.2636 val_acc 0.9116
epoch 3/3 batch_size 32 loss 0.2448 val_acc 0.9116
phase 2: fine-tuning layer4 and last layer
epoch 1/3 batch_size 32 loss 0.2855 val_acc 0.9216
epoch 2/3 batch_size 32 loss 0.1381 val_acc 0.9237
epoch 3/3 batch_size 32 loss 0.0850 val_acc 0.9230


## Step 5
Test the model on the test data, output the accuracy and save the model.

In [5]:
free_memory()
(preds, labels), _ = safe_call(lambda bs: predict_set(test_ds, bs), batch_size)
print(f"final test accuracy: {(preds == labels).float().mean().item():.4f}")
for c, name in enumerate(class_names):
    print(f"{name}: {(preds[labels == c] == c).float().mean().item():.4f}")
torch.save({"model": model.state_dict(), "classes": class_names}, "exp09_resnet18_intel.pt")

final test accuracy: 0.9300
buildings: 0.9130
forest: 0.9916
glacier: 0.8662
mountain: 0.8762
sea: 0.9725
street: 0.9701
